In [12]:
import os
os.environ["PATH"] += r";C:\Program Files\Tesseract-OCR;C:\poppler\Library\bin"
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files\Tesseract-OCR\tessdata"

import base64
from PIL import Image
import io
import json
from pprint import pprint
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_groq import ChatGroq 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()
print("✅ Imports done")

d:\Agentic AI\step3\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports done


In [13]:
# Cell 2 — Partition (ek baar chalao)
def partition_document(file_path: str):
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    print(f"✅ Partitioned — {len(elements)} elements found")
    return elements

file_path = "./docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1480.32it/s]


✅ Partitioned — 232 elements found


In [14]:
# Cell 3 — Chunking (baar baar chalao)
def create_chunks(elements):
    chunks = chunk_by_title(
        elements=elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks(elements)
chunks[12].metadata.orig_elements[0].to_dict()

✅ Created 32 chunks


{'type': 'Table',
 'element_id': 'a89281d85d3d589d6b9b2c889815cbce',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9282334446907043,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(561.3709716796875),
     np.float64(550.45751953125)),
    (np.float64(561.3709716796875), np.float64(908.8243408203125)),
    (np.float64(2395.14013671875), np.float64(908.8243408203125)),
    (np.float64(2395.14013671875), np.float64(550.45751953125))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-06-03T21:57:38',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Atten

In [15]:
def separate_content_types(chunk):
  """Analyze what types of content are in a chunk"""
  content_data = {
    'text': chunk.text,
    'table':[],
    'images':[],
    'types':['text']
  }

  if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
    for element in chunk.metadata.orig_elements:
      element_type = type(element).__name__

      if element_type == "Table":
        content_data["types"].append('table')
        table_html = getattr(element.metadata, 'text_as_html', element.text)
        content_data['table'].append(table_html)

      elif element_type == "Image":
        if hasattr(element, 'metadata') and hasattr(element.metadata, 'image'):
          content_data["types"].append('image')
          content_data['images'].append(element.metadata.image_base64)
    
  content_data['types'] = list(set(content_data['types']))

  return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images:List[str]) -> str:
  """Create AI-enhanced summary for mixed content"""
  try:
    
    llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0)

    prompt_text = f""" You are creating a searchable description for document content retrivel.

    CONTENT TO ANALYZE:
    TEXT CONTENT:
    {text}
    """

    if tables:
      prompt_text += "TABLES:\n"
      for i, table in enumerate(tables):
        prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """

          YOUR TASK
            Generate a comprehansive, searchable description that covers:

            1. Key facts, numbers, and data points from text and tables
            2. Main topics and concepts discussed
            3. Questions this content could answer
            4. Visual content analysis (charts, diagrams, patterns in images)
            5. Alternative search terms users might use

            Make it detailed and searchable - prioritize findability over brevity

            SEARCHABLE DESCRIPTION:" **
            """
        
    message_content = [{"type":"text","text":prompt_text}]

    for image_base64 in images:
      message_content.append({
        "type":"image_url",
        "image_url":{"url":f"data:image/jpeg;base64,{image_base64}"}
      })


    message = HumanMessage(content=message_content)
    response = llm.invoke([message])

    return response.content

  except Exception as e:
    print(f"     Ai Summary failed: {e}")

    summary = f"{text[:300]}..."

def summarise_chunks(chunks):
  """Process all chunks with AI Summaries"""
  print(" Processing chunks with AI Summaries...")

  langchain_documents = []
  total_chunks = len(chunks)


  for i, chunk in enumerate(chunks):
    current_chunk = i + 1
    print(f" Processing chunk {current_chunk}/{total_chunks}")

    content_data = separate_content_types(chunk)

    print(f" Types found: {content_data['types']}")
    print(f" Tables: {len(content_data['table'])}, Images: {len(content_data['images'])}")


    if content_data['table'] or content_data['images']:
      print(f"   -> Creating AI summary for mixed content..")
      try:
          enhaced_content = create_ai_enhanced_summary(
              content_data['text'],
              content_data['tables'],
              content_data['images']
          )
          print(f"  AI summary created successfully")
          print(f"   Enhanced content preview: {enhaced_content[:200]}")

      except Exception as e:
        print()
    
    else:
      print(f"  ->Using raw text (no tables/images)")
      enhaced_content = content_data['text']

    doc = Document(
          page_content=enhaced_content,
          metadata={
            "original_content": json.dumps({
              "raw_text":content_data['text'],
              "tables_html":content_data['table'],
              "images_base64":content_data['images']
            })
          }
        )
    langchain_documents.append(doc)

  print(f" Processed {len(langchain_documents)}chunks")

  print(langchain_documents)
  return langchain_documents

processed_chunks = summarise_chunks(chunks)


 Processing chunks with AI Summaries...
 Processing chunk 1/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 2/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 3/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 4/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 5/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 6/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 7/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 8/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 9/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Proce

In [32]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 32 chunks to chunks_export.json


In [17]:
def create_vector_store(document, persis_directory="db/chroma2_db"):
  """Create and persist ChromaDB vector store"""
  print("Create embedding and storing in ChromaDB...")

  embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

  print("--- Creating vector store ---")
  vectorstore = Chroma.from_documents(
    documents=document,
    embedding=embedding_model,
    persist_directory=persis_directory,
    collection_metadata={"hnsw:space":"cosine"}
  )

  print("--- Finished creating vector store ---")

  print(f" Vector store created and saved to {persis_directory}")

  return vectorstore

db = create_vector_store(processed_chunks)

Create embedding and storing in ChromaDB...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 778.54it/s]


--- Creating vector store ---
--- Finished creating vector store ---
 Vector store created and saved to db/chroma2_db


In [34]:
query = "What are the two main components of the Transformer architecture? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results. json")

✅ Exported 3 chunks to rag_results. json


[{'chunk_id': 1,
  'enhanced_content': '6.2 Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.\n\n8\n\nTable 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metrics are on the English-to-German translation development set, newstest2013. Listed perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to per-word perplexities.',
  'metadata': {'original_content': {'raw_text': '6.2 Model Variations\n\nTo evaluate the importance of different components of the Transformer, we varied our base model in different ways, measuring the change in performance on English-to-German translation on the\n\n5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for

In [18]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print(" Starting RAG Ingestion Pipeline")
    print("=" * 50)

    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks(elements)

    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
   
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persis_directory="dbv2/chroma_db")

    print("> Pipeline completed successfully!")
    return db

file_path = "./docs/attention-is-all-you-need.pdf"
run_complete_ingestion_pipeline(file_path)



 Starting RAG Ingestion Pipeline


No languages specified, defaulting to English.


✅ Partitioned — 232 elements found
✅ Created 32 chunks
 Processing chunks with AI Summaries...
 Processing chunk 1/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 2/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 3/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 4/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 5/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 6/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 7/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 8/32
 Types found: ['text']
 Tables: 0, Images: 0
  ->Using raw text (no tables/images)
 Processing chunk 9/32
 Types found: ['text']
 Tables: 0,

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1858.27it/s]


--- Creating vector store ---
--- Finished creating vector store ---
 Vector store created and saved to dbv2/chroma_db
> Pipeline completed successfully!


In [20]:
# Query the vector store
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

          CONTENT TO ANALYZE:
          """
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

The Transformer uses 8 attention heads in parallel, and the dimension of each head (dk and dv) is 64. This is stated in Document 3, where it says "In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64." This indicates that the model has 8 attention heads, each with a dimension of 64.
